In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "kirchhofer2012dogs")
original_data_pathway = os.path.join(pathway, "original_data")

# complete_path_1 = os.path.join(original_data_pathway, "Kirchhofer_2012__final trials tab_Data_Pointing_Study-ComprehensionII.csv")
complete_path_2 = os.path.join(original_data_pathway, "Kirchhofer_2012_all trials tab_Data_Pointing_Study-ComprehensionII.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

# df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)

df2["experiment_name"]="all_trials"
df2.columns
df2.rename(columns={"correct object": "correctobject"}, inplace=True)

In [3]:
code_list=[['A',"cup"],
           ['B','black_hose'],
           ['C','rope'],
           ['D','green_hose'],
           ['E','tube'],
           ['F','PVC'],
           ['G','spoon'],
           ['H','scraper']]
for x,y in code_list:
    df2.loc[df2.correctobject == x, ['correct_object']] = y

# df2.rename(columns={"correctobject": "correct_object"}, inplace=True)

In [4]:

df2.columns = map(str.lower, df2.columns)
df2=df2.applymap(lambda s: s.lower() if type(s) == str else s) 
fulldf=df2.rename(columns={"subject": "ape",
        "trialno.": "trial",
        "choosing which side?": "choosing_which_side",
        "opening correct side? (1=yes, 0=no)": "opening_correct_side",
        "position of correct object? (l=left,r=right)": "position_of_correct_object",
        "position of correct object?":"position_of_correct_object",
        "choosing which object?": "choosing_which_object",
        "giving object to e? (1=yes, 0=no)": "giving_object_to_e",
        "choosing which object?_codes":"choosing_which_object_codes",
        "tapeno.": "tape_number"})
fulldf['study_id']="kirchhofer2012dogs"


In [5]:
code_list=["position_of_correct_object"]
for index, x in enumerate(code_list):    
    fulldf[x] = fulldf[x].astype(str)
    temp=[]
    for entry in fulldf[x]:
        if entry == 'l':
            entry = "left"
        elif entry =='r':
            entry = "right"
        temp.append(entry)
    fulldf = fulldf.assign(temp_col=temp)
    fulldf=fulldf.rename(columns={'temp_col': x+'_codes'})

In [6]:
code_list=["opening_correct_side", "giving_object_to_e"]
for index, x in enumerate(code_list):    
    fulldf[x] = fulldf[x].astype(str)
    temp=[]
    for entry in fulldf[x]:
        if entry == '0' or entry =='0.0':
            entry = "no"
        elif entry =='1' or entry =='1.0':
            entry = "yes"
        temp.append(entry)
    fulldf = fulldf.assign(temp_col=temp)
    fulldf=fulldf.rename(columns={'temp_col': x+'_codes'})

In [7]:
temp_var = ''
out_list = []
for index, row in fulldf.iterrows():
    if not pd.isna(row['date']):
        temp_var = row['date']
    out_list.append(temp_var)
fulldf = fulldf.assign(date_temp=out_list)

fulldf[['day','month', 'year_temp']] = fulldf['date_temp'].str.split('/',expand=True)
year=[]
for index, row in fulldf.iterrows():
    if not pd.isna(row['year_temp']):
        year.append('20' + row['year_temp'])
    else:
        year.append("")
fulldf = fulldf.assign(year=year)


In [8]:
temp_var = ''
out_list = []
for index, row in fulldf.iterrows():
    if not pd.isna(row['age']):
        temp_var = row['age']
    out_list.append(temp_var)
fulldf = fulldf.assign(age_original=out_list)

In [9]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')

In [10]:
fulldf.replace('nan', np.nan, inplace=True)
# fulldf.columns

In [11]:
fulldf['comments'] = '  ' + fulldf['comments'].astype(str)

comment_replace_list = [[' s ', ' subject '],
                        [' e ', ' experimenter '],
                        ['bf', ' before focusing '],
                        [' h ', ' helper '],
                        ['  nan', np.nan],
                        [';', '']]
for x,y in comment_replace_list:
    fulldf['comments'].replace(x, y, inplace=True, regex=True)
# print(fulldf['comments'])

fulldf.rename(columns={"ape": "participant"}, inplace=True)


In [12]:

comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') #insert dob of participants
fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])##convert date of data collection to datetime format
fulldf['dob'] = pd.to_datetime(fulldf['dob'])##convert date of birth to datetime format

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

In [13]:
fulldf=fulldf[['study_id','year', 'month', 'day',
       'participant', 'age_original', 'age_in_years', 'sex','species',
       'session', 'trial', 'excluded_trials', 'repetition', 'correct_object',
        'position_of_correct_object_codes','opening_correct_side_codes',  
       'giving_object_to_e_codes']]
fulldf.columns =fulldf.columns.str.replace('_codes', '')

In [14]:

comp_out_path_stand = os.path.join(out_pathway, 'kirchhofer2012dogs_standardized.csv')
fulldf.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

##for one glossary
names = fulldf.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'kirchhofer2012dogs_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)